# FinanceFlow Underwriting Data Contract
## Part B â€” Great Expectations Checkpoint


In [ ]:
import pandas as pd
import great_expectations as gx
import great_expectations.expectations as gxe

df = pd.read_csv("financeflow_raw.csv")
print(f"Loaded {len(df)} rows")


In [ ]:
# Create GX context and datasource
context = gx.get_context(mode="ephemeral")
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset("financeflow_raw")

batch_definition = data_asset.add_batch_definition("raw_data", parameters={"dataframe": df})
batch = batch_definition.get_batch()


In [ ]:
# Rule 1: Underwriting requires a valid credit score â€” missing scores block model inference.
expectation_1 = gxe.ExpectColumnValuesToNotBeNull(column="credit_score")
result_1 = batch.validate(expectation_1)
print(f"1. credit_score not null: {'PASS' if result_1['success'] else 'FAIL'} (unexpected: {result_1['result']['unexpected_count']})")

# Rule 2: FICO range is 300â€“850; values outside this are data errors.
expectation_2 = gxe.ExpectColumnValuesToBeBetween(column="credit_score", min_value=300, max_value=850)
result_2 = batch.validate(expectation_2)
print(f"2. credit_score 300-850: {'PASS' if result_2['success'] else 'FAIL'} (unexpected: {result_2['result']['unexpected_count']})")

# Rule 3: Without the target label, the record cannot contribute to any risk model.
expectation_3 = gxe.ExpectColumnValuesToNotBeNull(column="default_flag")
result_3 = batch.validate(expectation_3)
print(f"3. default_flag not null: {'PASS' if result_3['success'] else 'FAIL'} (unexpected: {result_3['result']['unexpected_count']})")

# Rule 4: Revenue cannot be negative â€” indicates data entry error, not viable business.
expectation_4 = gxe.ExpectColumnValuesToBeBetween(column="annual_revenue_usd", min_value=0)
result_4 = batch.validate(expectation_4)
print(f"4. annual_revenue_usd >= 0: {'PASS' if result_4['success'] else 'FAIL'} (unexpected: {result_4['result']['unexpected_count']})")

# Rule 5: DTI below 0.05 is unrealistic; above 1.00 means the business cannot cover debts.
expectation_5 = gxe.ExpectColumnValuesToBeBetween(column="debt_to_income_ratio", min_value=0.05, max_value=1.00)
result_5 = batch.validate(expectation_5)
print(f"5. DTI 0.05-1.00: {'PASS' if result_5['success'] else 'FAIL'} (unexpected: {result_5['result']['unexpected_count']})")

# Rule 6: Sector must be one of the 7 recognized industry categories.
expectation_6 = gxe.ExpectColumnValuesToBeInSet(column="sector", value_set=["Retail", "Manufacturing", "Technology", "Hospitality", "Healthcare", "Construction", "Agriculture"])
result_6 = batch.validate(expectation_6)
print(f"6. sector in valid set: {'PASS' if result_6['success'] else 'FAIL'} (unexpected: {result_6['result']['unexpected_count']})")


## Data Contract Status

**Data Contract Status â€” FinanceFlow Underwriting**
- **Credit scores:** 30 applications arrived without a credit score, which means the model cannot generate a risk assessment for them. These have been quarantined and the data operations team has been notified to pull scores from the core banking system.
- **Revenue records:** 15 applications show negative revenue, which appears to be a data entry error from the export process. The credit team should verify the original loan application PDFs for these records before any decision is made.
- **Target labels:** 25 historical applications are missing their default outcome flag, making them unusable for model training. This delays the model build until the servicing team can manually code these flags from the collections system.

All other checks â€” FICO range, debt-to-income limits, and sector codes â€” passed without issues.
